In [ ]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)
library(cowplot)
library(hexbin)
library(repr)
library(reshape2)
library(scico)
#library(carto)
options(repr.plot.width=10, repr.plot.height=8)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

library(MASS)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("old", "young") #all

for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c( # "MATES"="#F98A7B",
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#F3E088",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)


colorSamples <- c("old"="#7A646A",
                "young"="#A3C6E3",
                "all"="#8190a9")

thrMinCells <- 500 * 0.05

theme_set(
  theme_minimal(base_size = 20)
)

In [ ]:
# import workspace
load("workspaces/mm_RA_simulation_evaluation_objectCreation.Rdata")


In [ ]:
# function to expand matrix to certain list of rows and columns

expand_numeric_matrix <- function(M, rows, cols) {
  out <- Matrix(0, nrow = length(rows), ncol = length(cols), sparse = TRUE)
  rownames(out) <- rows
  colnames(out) <- cols

  out[rownames(M), colnames(M)] <- M
  out
}


# Expression of FPs

## Raw counts

In [ ]:
# get all FP counts = counts that are detected in a cell where a locus is not expressed

layer <- "counts"

FPcountList <- list()
TPcountList <- list()

for(sample in samples){

    splatter_obj <- splatter_objs[[sample]]

    FPcountList[[sample]] <- NULL

    for(tool in names(objList)){

        obj <- objList[[tool]][[sample]]

        splatterMat <- GetAssayData(splatter_obj, layer = layer)
        objMat <- GetAssayData(obj, layer = layer)

        # #avg raw expression per cell 
        # avgExprCells <- colMeans(splatterMat)

        # #avg raw expression per cell 
        # totCountsCells <- colSums(splatterMat)

        # 1. Get full union of rows and columns
        all_rows <- union(rownames(splatterMat), rownames(objMat))
        all_cols <- union(colnames(splatterMat), colnames(objMat))

        splatterMat_exp <- expand_numeric_matrix(splatterMat, all_rows, all_cols)
        objMat_exp <- expand_numeric_matrix(objMat, all_rows, all_cols)

        # Compute False Positives
        FP_mask <- (objMat_exp > 0) & (splatterMat_exp == 0)

        ij <- which(FP_mask, arr.ind = TRUE)

        fp_df <- data.frame(
                feature = rownames(objMat_exp)[ij[,1]],
                cell    = colnames(objMat_exp)[ij[,2]],
                value   = objMat_exp[ij],
                row.names = NULL
            )

        colnames(fp_df) <- c("feature", "cell", "value")
        fp_df$tool <- tool
        fp_df$age <- sample
        FPcountList[[sample]][[tool]] <- fp_df

                # Compute True Positives
        TP_mask <- (objMat_exp > 0) & (splatterMat_exp > 0)
        ij <- which(TP_mask, arr.ind = TRUE)
        tp_df <- data.frame(
                feature = rownames(objMat_exp)[ij[,1]],
                cell    = colnames(objMat_exp)[ij[,2]],
                value   = objMat_exp[ij],
                row.names = NULL
            )
        colnames(tp_df) <- c("feature", "cell", "value")
        tp_df$tool <- tool
        tp_df$age <- sample
        TPcountList[[sample]][[tool]] <- tp_df

    }
}


In [ ]:
rawcounts <- GetAssayData(splatter_objs[["young"]], layer="counts")
plot(density(colSums(rawcounts)))
plot(density(colMeans(rawcounts)))
head(colSums(rawcounts))

In [ ]:
FPcountDf <- list()
TPcountDf <- list()
FPvsTPcountDf <- list()
for(sample in samples){
    FPcountDf[[sample]] <- do.call(rbind, FPcountList[[sample]])
    FPcountDf[[sample]]$type <- "FP"
    TPcountDf[[sample]] <- do.call(rbind, TPcountList[[sample]])
    TPcountDf[[sample]]$type <- "TP"
    FPvsTPcountDf[[sample]] <- rbind(TPcountDf[[sample]], FPcountDf[[sample]])

}


In [ ]:
FPcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[FPcountDf[["young"]]$cell]
FPcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[FPcountDf[["young"]]$cell]

FPcountDf[["young"]]$FPoverAvgSim <- FPcountDf[["young"]]$value / FPcountDf[["young"]]$avgExpr

FPcountDf[["young"]]$normOverSim <- FPcountDf[["young"]]$value / FPcountDf[["young"]]$totCounts * 10000



In [ ]:
FPvsTPcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[FPvsTPcountDf[["young"]]$cell]
FPvsTPcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[FPvsTPcountDf[["young"]]$cell]

FPvsTPcountDf[["young"]]$FPoverAvgSim <- FPvsTPcountDf[["young"]]$value / FPvsTPcountDf[["young"]]$avgExpr

FPvsTPcountDf[["young"]]$normOverSim <- FPvsTPcountDf[["young"]]$value / FPvsTPcountDf[["young"]]$totCounts * 10000



In [ ]:
TPcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[TPcountDf[["young"]]$cell]
TPcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[TPcountDf[["young"]]$cell]

TPcountDf[["young"]]$FPoverAvgSim <- TPcountDf[["young"]]$value / TPcountDf[["young"]]$avgExpr

TPcountDf[["young"]]$normOverSim <- TPcountDf[["young"]]$value / TPcountDf[["young"]]$totCounts * 10000

In [ ]:
FPcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FPoverAvgSim = median(FPoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
FPcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FPoverAvgSim = median(FPoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
TPcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FPoverAvgSim = median(FPoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
options(repr.plot.width=7, repr.plot.height=4.5)

#for(sample in samples){
  sample <- "young"
    #reorder tools
    FPcountDf[[sample]]$tool <- factor(FPcountDf[[sample]]$tool, levels=rev(names(colorTools)))

    show(ggplot(FPcountDf[[sample]], aes(y=tool, x=value, fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw counts of FP ", sample, " TEs")) +
        coord_trans(x = "sqrt") +
        guides(fill="none") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=10), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))

    show(ggplot(FPcountDf[[sample]], aes(y=tool, x=log2(value+1), fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw counts of FP ", sample, " TEs")) +
        guides(fill="none") +
        xlab("log2 count") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=10), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))

    show(ggplot(FPcountDf[[sample]], aes(y=tool, x=normOverSim, fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw FP count / tot counts of ", sample, " TEs")) +
        guides(fill="none") + 
        coord_trans(x = "sqrt") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=18), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FP_expression.pdf"), device="pdf", width=7, height=4.5)
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FP_expression.png"), device="png", width=7, height=4.5)
#}


In [ ]:
options(repr.plot.width=8, repr.plot.height=6.5)

#for(sample in samples){
  sample <- "young"
    #reorder tools
    FPvsTPcountDf[[sample]]$tool <- factor(FPvsTPcountDf[[sample]]$tool, levels=rev(names(colorTools)))

    show(ggplot(FPvsTPcountDf[[sample]], aes(y=tool, x=normOverSim, fill=type)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7, ) +
        #scale_fill_scico_d(palette="berlin",direction = -1) +
        scale_fill_viridis_d()+
        xlim(c(0, max(FPvsTPcountDf[[sample]]$normOverSim))) +
        theme_pubclean() + ggtitle(paste0("FP vs TP counts of ", sample, " TEs"), subtitle="Normalized by tot simuated counts") +
        coord_trans(x = "sqrt") +
        theme(text=element_text(size=20), 
          plot.title = element_text(size=20, hjust=0.5), 
          plot.subtitle = element_text(size=20, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=18)
        ))
  ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FP_vs TP_expression.pdf"), device="pdf", width=7, height=4.5)
  ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FP_vs TP_expression.pdf"), device="pdf", width=7, height=4.5)

#}


In [ ]:
df_cell_avg <- FPvsTPcountDf[[sample]] %>%
  group_by(tool, cell, type) %>%
  summarise(
    avg_count = mean(value, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_wider(
    names_from = type,
    values_from = avg_count
  ) %>%
  filter(!is.na(TP), !is.na(FP)) %>%   # both must exist at cell level
  mutate(
    FP_TP_avg_ratio = FP / TP,
    log2_FC = log2((FP + 1e-6) / (TP + 1e-6))
  )


sample <- "young"
  #reorder tools
  df_cell_avg$tool <- factor(df_cell_avg$tool, levels=rev(names(colorTools)))

options(repr.plot.width=8, repr.plot.height=4.5)

  show(ggplot(df_cell_avg, aes(y=tool, x=FP_TP_avg_ratio, fill=tool)) + 
      geom_boxplot(outlier.size=0.5, linewidth=0.7, alpha = 0.6) +
      scale_fill_manual(values=colorTools) +
      theme_pubclean() + ggtitle(paste0("Fold-change of FP vs TP counts of ", sample, " TEs"), subtitle="per cell") +
      #coord_trans(x = "sqrt") +
      guides(fill="none") + 
      theme(text=element_text(size=20), 
        plot.title = element_text(size=20, hjust=0.5), 
        plot.subtitle = element_text(size=20, hjust=0.5), 
        axis.text.y = element_text(size=20), 
        axis.text.x = element_text(size=18)
      ))
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FoldChange_FP_vs TP_expression.pdf"), device="pdf", width=7, height=4.5)
    
    #violin
    show(ggplot(df_cell_avg, aes(y=tool, x=FP_TP_avg_ratio, fill=tool)) + 
      geom_violin(outlier.size=0.5, linewidth=0.7, alpha=0.6) +
      scale_fill_manual(values=colorTools) +
      theme_pubclean() + ggtitle(paste0("Fold-change of FP vs TP counts of ", sample, " TEs"), subtitle="per cell") +
      #coord_trans(x = "sqrt") +
      guides(fill="none") + 
      theme(text=element_text(size=20), 
        plot.title = element_text(size=20, hjust=0.5), 
        plot.subtitle = element_text(size=20, hjust=0.5), 
        axis.text.y = element_text(size=20), 
        axis.text.x = element_text(size=18)
      ))


## Normalized counts

In [ ]:
# get all FP counts = counts that are detected in a cell where a locus is not expressed

layer <- "data"

FPcountList <- list()

for(sample in samples){

    splatter_obj <- splatter_objs[[sample]]

    FPcountList[[sample]] <- NULL

    for(tool in names(objList)){

        obj <- objList[[tool]][[sample]]

        splatterMat <- GetAssayData(splatter_obj, layer = layer)
        objMat <- GetAssayData(obj, layer = layer)

        library(Matrix)

        # 1. Get full union of rows and columns
        all_rows <- union(rownames(splatterMat), rownames(objMat))
        all_cols <- union(colnames(splatterMat), colnames(objMat))


        splatterMat_exp <- expand_numeric_matrix(splatterMat, all_rows, all_cols)
        objMat_exp <- expand_numeric_matrix(objMat, all_rows, all_cols)

        # Compute False Positives
        FP_mask <- (objMat_exp > 0) & (splatterMat_exp == 0)

        ij <- which(FP_mask, arr.ind = TRUE)

        fp_df <- data.frame(
                feature = rownames(objMat_exp)[ij[,1]],
                cell    = colnames(objMat_exp)[ij[,2]],
                value   = objMat_exp[ij],
                row.names = NULL
            )

        colnames(fp_df) <- c("feature", "cell", "value")
        fp_df$tool <- tool
        fp_df$age <- sample
        FPcountList[[sample]][[tool]] <- fp_df

    }
}


In [ ]:
FPcountDf <- list()

for(sample in samples){
    FPcountDf[[sample]] <- do.call(rbind, FPcountList[[sample]])

}


In [ ]:
options(repr.plot.width=7, repr.plot.height=4.5)

for(sample in samples){
    #reorder tools
    FPcountDf[[sample]]$tool <- factor(FPcountDf[[sample]]$tool, levels=rev(names(colorTools)))

    show(ggplot(FPcountDf[[sample]], aes(y=tool, x=value, fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Expression of FP ", sample, " TEs")) +
        guides(fill="none") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=18), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FP_expression.pdf"), device="pdf", width=7, height=4.5)
}


# Expression of FNs

## Raw counts

In [ ]:
# get all FN counts = counts that are NOT detected in a cell where a locus IS expressed

layer <- "counts"

FNcountList <- list()
TPcountList <- list()

for(sample in samples){

    splatter_obj <- splatter_objs[[sample]]

    FNcountList[[sample]] <- NULL

    for(tool in names(objList)){

        obj <- objList[[tool]][[sample]]

        splatterMat <- GetAssayData(splatter_obj, layer = layer)
        objMat <- GetAssayData(obj, layer = layer)

        # #avg raw expression per cell 
        # avgExprCells <- colMeans(splatterMat)

        # #avg raw expression per cell 
        # totCountsCells <- colSums(splatterMat)

        # 1. Get full union of rows and columns
        all_rows <- union(rownames(splatterMat), rownames(objMat))
        all_cols <- union(colnames(splatterMat), colnames(objMat))

        splatterMat_exp <- expand_numeric_matrix(splatterMat, all_rows, all_cols)
        objMat_exp <- expand_numeric_matrix(objMat, all_rows, all_cols)

        # Compute False Positives
        FN_mask <- (objMat_exp == 0) & (splatterMat_exp > 0)

        ij <- which(FN_mask, arr.ind = TRUE)

        fn_df <- data.frame(
                feature = rownames(splatterMat_exp)[ij[,1]],
                cell    = colnames(splatterMat_exp)[ij[,2]],
                value   = splatterMat_exp[ij],
                row.names = NULL
            )

        colnames(fn_df) <- c("feature", "cell", "value")
        fn_df$tool <- tool
        fn_df$age <- sample
        FNcountList[[sample]][[tool]] <- fn_df

                # Compute True Positives
        TP_mask <- (objMat_exp > 0) & (splatterMat_exp > 0)
        ij <- which(TP_mask, arr.ind = TRUE)
        tp_df <- data.frame(
                feature = rownames(splatterMat_exp)[ij[,1]],
                cell    = colnames(splatterMat_exp)[ij[,2]],
                value   = splatterMat_exp[ij],
                row.names = NULL
            )
        colnames(tp_df) <- c("feature", "cell", "value")
        tp_df$tool <- tool
        tp_df$age <- sample
        TPcountList[[sample]][[tool]] <- tp_df

    }
}


In [ ]:
FNcountDf <- list()
TPcountDf <- list()
FNvsTPcountDf <- list()
for(sample in samples){
    FNcountDf[[sample]] <- do.call(rbind, FNcountList[[sample]])
    FNcountDf[[sample]]$type <- "FN"
    TPcountDf[[sample]] <- do.call(rbind, TPcountList[[sample]])
    TPcountDf[[sample]]$type <- "TP"
    FNvsTPcountDf[[sample]] <- rbind(TPcountDf[[sample]], FNcountDf[[sample]])

}


In [ ]:
FNcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[FNcountDf[["young"]]$cell]
FNcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[FNcountDf[["young"]]$cell]

FNcountDf[["young"]]$FNoverAvgSim <- FNcountDf[["young"]]$value / FNcountDf[["young"]]$avgExpr

FNcountDf[["young"]]$normOverSim <- FNcountDf[["young"]]$value / FNcountDf[["young"]]$totCounts * 10000



In [ ]:
FNvsTPcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[FNvsTPcountDf[["young"]]$cell]
FNvsTPcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[FNvsTPcountDf[["young"]]$cell]

FNvsTPcountDf[["young"]]$FNoverAvgSim <- FNvsTPcountDf[["young"]]$value / FNvsTPcountDf[["young"]]$avgExpr

FNvsTPcountDf[["young"]]$normOverSim <- FNvsTPcountDf[["young"]]$value / FNvsTPcountDf[["young"]]$totCounts * 10000



In [ ]:
TPcountDf[["young"]]$avgExpr <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=median)[TPcountDf[["young"]]$cell]
TPcountDf[["young"]]$totCounts <- apply(X=GetAssayData(splatter_objs[["young"]], layer="counts"),MARGIN = 2, FUN=sum)[TPcountDf[["young"]]$cell]

TPcountDf[["young"]]$FNoverAvgSim <- TPcountDf[["young"]]$value / TPcountDf[["young"]]$avgExpr

TPcountDf[["young"]]$normOverSim <- TPcountDf[["young"]]$value / TPcountDf[["young"]]$totCounts * 10000

In [ ]:
FNcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FNoverAvgSim = median(FNoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
FNcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FNoverAvgSim = median(FNoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
TPcountDf[["young"]] %>%
  group_by(tool) %>%
  summarise(
    median_value = median(value, na.rm = TRUE),
    median_avgExpr = median(avgExpr, na.rm = TRUE),
    median_FNoverAvgSim = median(FNoverAvgSim, na.rm = TRUE),
    median_normOverSim = median(normOverSim, na.rm = TRUE)
  )

In [ ]:
options(repr.plot.width=7, repr.plot.height=4.5)

#for(sample in samples){
  sample <- "young"
    #reorder tools
    FNcountDf[[sample]]$tool <- factor(FNcountDf[[sample]]$tool, levels=rev(names(colorTools)))

    show(ggplot(FNcountDf[[sample]], aes(y=tool, x=value, fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw counts of FN ", sample, " TEs")) +
        coord_trans(x = "sqrt") +
        guides(fill="none") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=10), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))

    show(ggplot(FNcountDf[[sample]], aes(y=tool, x=log2(value+1), fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw counts of FN ", sample, " TEs")) +
        guides(fill="none") +
        xlab("log2 count") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=10), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))

    show(ggplot(FNcountDf[[sample]], aes(y=tool, x=normOverSim, fill=tool)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        scale_fill_manual(values=colorTools) +
        theme_pubclean() + ggtitle(paste0("Raw FN count / tot counts of ", sample, " TEs")) +
        guides(fill="none") + 
        coord_trans(x = "sqrt") +
        theme(text=element_text(size=20), 
        #   plot.title = element_text(size=18, hjust=0.5), 
        #   plot.subtitle = element_text(size=18, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=18), 
        #   strip.text = element_text(size=17, hjust=0.5, vjust = 0.5),
        #strip.background = element_rect(fill="#F4F1EE", linewidth = 3, color = "white"),
        #panel.spacing = unit(1, "lines")
        
        ))
 #   ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FN_expression.pdf"), device="pdf", width=7, height=4.5)
#}


In [ ]:
options(repr.plot.width=8, repr.plot.height=6.5)

#for(sample in samples){
  sample <- "young"
    #reorder tools
    FNvsTPcountDf[[sample]]$tool <- factor(FNvsTPcountDf[[sample]]$tool, levels=rev(names(colorTools)))

    show(ggplot(FNvsTPcountDf[[sample]], aes(y=tool, x=normOverSim, fill=type)) + 
        geom_boxplot(outlier.size=0.5, linewidth=0.7) +
        #scale_fill_scico_d(palette="berlin",direction = -1) +
        scale_fill_viridis_d()+
        xlim(c(0, max(FNvsTPcountDf[[sample]]$normOverSim))) +
        theme_pubclean() + ggtitle(paste0("FN vs TP counts of ", sample, " TEs")) +
        coord_trans(x = "sqrt") +
        theme(text=element_text(size=20), 
          plot.title = element_text(size=20, hjust=0.5), 
          plot.subtitle = element_text(size=20, hjust=0.5), 
          axis.text.y = element_text(size=20), 
          axis.text.x = element_text(size=18)
        ))
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FN_vs TP_expression.png"), device="png", width=7, height=4.5)
  ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FN_vs TP_expression.pdf"), device="pdf", width=7, height=4.5)
#}


In [ ]:
df_cell_avg <- FNvsTPcountDf[[sample]] %>%
  group_by(tool, cell, type) %>%
  summarise(
    avg_count = mean(value, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_wider(
    names_from = type,
    values_from = avg_count
  ) %>%
  filter(!is.na(TP), !is.na(FN)) %>%   # both must exist at cell level
  mutate(
    FN_TP_avg_ratio = FN / TP,
    log2_FC = log2((FN + 1e-6) / (TP + 1e-6))
  )


sample <- "young"
  #reorder tools
  df_cell_avg$tool <- factor(df_cell_avg$tool, levels=rev(names(colorTools)))

options(repr.plot.width=8, repr.plot.height=4.5)

  show(ggplot(df_cell_avg, aes(y=tool, x=FN_TP_avg_ratio, fill=tool)) + 
      geom_boxplot(outlier.size=0.5, linewidth=0.7, alpha = 0.6) +
      scale_fill_manual(values=colorTools) +
      theme_pubclean() + ggtitle(paste0("Fold-change of FN vs TP counts of ", sample, " TEs"), subtitle="per cell") +
      #coord_trans(x = "sqrt") +
      guides(fill="none") + 
      theme(text=element_text(size=20), 
        plot.title = element_text(size=20, hjust=0.5), 
        plot.subtitle = element_text(size=20, hjust=0.5), 
        axis.text.y = element_text(size=20), 
        axis.text.x = element_text(size=18)
      ))
    ggsave(paste0("figures_", dataset_id, "_", sample, "/boxplot_FoldChange_FN_vs TP_expression.pdf"), device="pdf", width=7, height=4.5)
    
    #violin
    show(ggplot(df_cell_avg, aes(y=tool, x=FN_TP_avg_ratio, fill=tool)) + 
      geom_violin(outlier.size=0.5, linewidth=0.7, alpha=0.6) +
      scale_fill_manual(values=colorTools) +
      theme_pubclean() + ggtitle(paste0("Fold-change of FN vs TP counts of ", sample, " TEs"), subtitle="per cell") +
      #coord_trans(x = "sqrt") +
      guides(fill="none") + 
      theme(text=element_text(size=20), 
        plot.title = element_text(size=20, hjust=0.5), 
        plot.subtitle = element_text(size=20, hjust=0.5), 
        axis.text.y = element_text(size=20), 
        axis.text.x = element_text(size=18)
      ))
